# Smart Upload Gateway: Edge Actions auf uploadp


mTLS authentifiziert den Client am bestehenden Endpunkt `cptmtls-uploadp`.
Front Door authentifiziert sich mit Managed Identity am Blob Storage.
Der Endpunkt `cptmtls-uploadp` ist ueber `uploadp.cptdaz.org` erreichbar (Azure-DNS-Zone `cptdaz.org`).
Der Workflow `afd-uploadp` deployt Domain, Edge Action und Route; dieses Notebook zeigt dasselbe Schritt fuer Schritt.
Beide Routen nutzen denselben mTLS-Endpunkt, dieselbe Managed Identity und denselben Storage-Origin.
Nur der neue Pfad `/upload/edge/*` verwendet die Edge Action; `/upload/*` bleibt unveraendert.
Es gibt keinen Rewrite: `/upload/edge/datei.json` wird im Container `upload` als `edge/datei.json` gespeichert.

Voraussetzungen: Azure CLI ist angemeldet, die Basis und `afd-uploadp` sind deployt,
Clientzertifikat und Schluessel liegen lokal vor. Benoetigt werden Bash, HTTPie,
jq, Azure CLI, AzCopy, OpenSSL, Python 3, unzip und Node.js. Waehle den Bash-Kernel aus `.venv`.
Alle Codezellen sind Shell Script und werden der Reihe nach ausgefuehrt.
Bearbeite die zentrale Konfiguration `edge-upload.env.sh`, bevor du die erste Codezelle ausfuehrst.
Die Ausgabe zeigt Konfiguration und Dateipfade, niemals private Schluessel oder Tokens.

Die Deploymentzelle erstellt eine Edge Action und eine separate Route in Azure.
Testdateien erhalten einen eindeutigen Namen. Die letzten Zellen entfernen nur
die Dateien dieses Laufs; Action und Route bleiben bestehen. Ausgaben nicht einchecken.

> **Wichtig: Diese Edge Action ist keine Sicherheitsgrenze, sondern eine Vorpruefung.**
> Edge Action Code darf hoechstens 10 ms laufen. Ueberschreitet er das Limit, stoppt Front Door ihn
> ohne Fehler und verarbeitet die Anfrage **ohne Action weiter (fail-open)**: eine Ablehnung findet dann nicht statt.
> Front Door lehnt ausserdem nur mit 401 oder 403 ab, andere Statuscodes werden ignoriert.
>
> Messung in diesem Setup: 200 GET-Anfragen und 200 PUT-Anfragen mit allen Pruefungen wurden jeweils
> zu 100 % abgelehnt, es entstand kein Blob. Das ist eine Stichprobe bei geringer Last, keine Garantie
> fuer Lastspitzen oder andere Standorte.
>
> Geeignet: Komfortpruefungen (Content-Type, Endung, deklarierte Groesse), Header-Normalisierung und Metadaten.
> Nicht geeignet: Authentifizierung, Autorisierung und Schutz vor Missbrauch. Das leisten mTLS, Storage-Berechtigungen
> (SAS, Managed Identity, RBAC) und die WAF (siehe Notebook upload-ja4), die nicht uebersprungen werden koennen.
>
> Geraete-IDs sind Clientangaben.
> Dateiendung, Content-Type und deklarierte Groesse sind keine Inhalts- oder Malwarepruefung.
> Direkter Storage-Zugriff und der Vergleichspfad umgehen diese zusaetzliche Qualitaetspruefung.


## Policy der Edge Action

Die [JavaScript-Policy](../edge/upload-policy.js) akzeptiert nur einzelne `PUT`-Requests (BlockBlob) ohne Query-Parameter. Pflicht-Header sind `X-Device-Id` (1 bis 64 ASCII-Buchstaben, Ziffern, `_` oder `-`, Beginn mit Buchstabe oder Ziffer) und `X-Upload-Kind`:

| Upload-Kind   | Endung | Content-Type       | Maximale deklarierte Content-Length |
|---------------|--------|--------------------|-------------------------------------|
| `telemetry`   | `.json`| `application/json` | 1 MiB                               |
| `diagnostics` | `.zip` | `application/zip`  | 20 MiB                              |

* Dateinamen beginnen mit Buchstabe oder Ziffer und enthalten maximal 128 ASCII-Buchstaben, Ziffern, Punkte, `_` oder `-`. Unterverzeichnisse sind nicht erlaubt.
* Akzeptierte Requests erhalten `x-ms-meta-upload_kind`, `x-ms-meta-device_id` und `x-ms-meta-policy_version: v1`. Vom Client gesendete Werte dieser Header werden ueberschrieben.
* Abgelehnte Requests erhalten **HTTP 403** mit `x-upload-policy: v1` und dem Grund in `x-upload-error` (zum Beispiel `file-type-mismatch`). Es gibt keinen Antwortkoerper. Front Door beendet eine Anfrage in einer Edge Action nur mit 401 oder 403; jeder andere Statuscode (405, 413, 415 ...) gilt als ungueltig und die Anfrage laeuft einfach weiter. Deshalb unterscheidet der Header `x-upload-error` die Gruende und nicht der Statuscode.

## Deployment ausserhalb des Notebooks

Nach `afd-uploadp` genuegt `bash projects/mtlsupload/scripts/deploy-upload-edge.sh`. Das Skript benennt jede Code-Version nach ihrem SHA-256, wartet auf die Validierung, laedt das deployte ZIP herunter und vergleicht das JavaScript Byte fuer Byte mit der lokalen Quelle. Erst danach wird die Version mit `swapDefault` aktiviert und die separate Route angehaengt. Ein erneuter Lauf mit unveraendertem Code verwendet dieselbe Version. Azure erlaubt hoechstens drei Versionen: Vor der vierten Code-Revision muss eine ungenutzte, nicht aktive Version entfernt werden. Die Propagation in Front Door kann einige Minuten dauern.

Lokale Tests der Policy: `node --test projects/mtlsupload/edge/upload-policy.test.cjs`.

Das Notebook prueft vor dem Deployment, dass die Custom Domain freigegeben ist und per DNS aufgeloest wird. Es wartet auf eine von der Policy erzeugte HTTP-403-Antwort mit `x-upload-policy: v1` am Live-PoP, bevor es hochlaedt: ein erfolgreiches ARM-Provisioning allein beweist nicht, dass die Action ausgefuehrt wird. AzCopy meldet sich direkt am Storage an (Azure CLI), nicht ueber den mTLS-Endpunkt; der angemeldete Benutzer braucht Blob-Datenrechte. Weitere Informationen: [Edge Actions in Azure Front Door](https://learn.microsoft.com/azure/frontdoor/edge-actions).


In [ ]:
set -o pipefail
source ./edge-upload.env.sh &&
mkdir -p "$DEMO_DIR" &&
env | grep -E "^(EDGE_REPO_ROOT|AZURE_SUBSCRIPTION_ID|RESOURCE_GROUP|PROFILE|ENDPOINT|HOST|DOMAIN|STORAGE|CONTAINER|EDGE_ACTION|EDGE_VERSION|EDGE_API|AFD_API|CERT|KEY|DEVICE_ID|RUN_ID|DEMO_DIR|TELEMETRY_FILE|DIAGNOSTICS_FILE|OVERSIZE_FILE|TELEMETRY_BLOB|DIAGNOSTICS_BLOB|BASELINE_BLOB|EDGE_URL|ARM_ROOT|VERSION_URL|AZCOPY_AUTO_LOGIN_TYPE|AZCOPY_LOG_LOCATION|AZCOPY_JOB_PLAN_LOCATION|PAGER)=" | sort

## Werkzeuge, Konto und Zertifikat

In [ ]:
http --version &&
jq --version &&
az version &&
azcopy --version &&
python3 --version &&
node --version


In [ ]:
command -v unzip sha256sum

In [ ]:
az account show --subscription "$AZURE_SUBSCRIPTION_ID" --query '{name:name,id:id}' -o table &&
openssl verify -CAfile "$EDGE_REPO_ROOT/projects/mtlsupload/pki/ca.crt" "$CERT" &&
test -r "$KEY"


## Bestehenden Endpunkt pruefen

Die Custom Domain muss bereits validiert, ausgerollt und per DNS erreichbar sein.
Diese Voraussetzung gehoert zum vorhandenen afd-uploadp-Deployment.
Bei Pending zuerst dessen DNS- und Domainvalidierung abschliessen.
Die Pruefung bricht vor Aenderungen in Azure ab, wenn die Basis noch nicht bereit ist.

In [ ]:
az rest --subscription "$AZURE_SUBSCRIPTION_ID" --method get \
  --url "$ARM_ROOT/profiles/$PROFILE/customDomains/$DOMAIN?api-version=$AFD_API" |
jq -e '{host: .properties.hostName, validation: .properties.domainValidationState, deployment: .properties.deploymentStatus} | ., (.validation == "Approved" and .deployment == "Succeeded")' &&
getent hosts "$HOST"


## Policy lokal testen und deployen

Die Tests pruefen auch exakte Groessengrenzen, fehlende oder ungueltige Content-Length,
falsche Methoden, Query-Operationen und manipulierte Metadaten.
Das Deployment verwendet eine aus dem Code-Hash benannte unveraenderliche Version,
wartet auf Codevalidierung, vergleicht den hochgeladenen Code bytegenau und
aktiviert dann Version und Route. Unveraenderter Code verwendet dieselbe Version.
Bei drei vorhandenen Versionen zuerst eine nicht mehr benoetigte Nicht-Default-Version
entfernen, bevor eine weitere Coderevision deployt wird. Es kann mehrere Minuten dauern.
Bei noch laufender Front-Door-Propagation die Uploadzellen spaeter erneut ausfuehren.

In [ ]:
node --test "$EDGE_REPO_ROOT/projects/mtlsupload/edge/upload-policy.test.cjs"

In [ ]:
bash "$EDGE_REPO_ROOT/projects/mtlsupload/scripts/deploy-upload-edge.sh"

## Konfiguration in Azure nachweisen

In [ ]:
az rest --subscription "$AZURE_SUBSCRIPTION_ID" --method get --url "$VERSION_URL" |
jq -e '{name, properties} | select(.properties.provisioningState == "Succeeded" and .properties.validationStatus == "Succeeded")'


In [ ]:
az rest --subscription "$AZURE_SUBSCRIPTION_ID" --method get \
  --url "$ARM_ROOT/profiles/$PROFILE/afdEndpoints/$ENDPOINT/routes?api-version=$AFD_API" |
jq '.value[] | {name, patterns: .properties.patternsToMatch, ruleSets: .properties.ruleSets}'


In [ ]:
az rest --subscription "$AZURE_SUBSCRIPTION_ID" --method get \
  --url "$ARM_ROOT/profiles/$PROFILE/ruleSets/uploadedge/rules/uploadpolicy?api-version=$AFD_API" |
jq -e '.properties.actions[] | select(.name == "EdgeAction")'


## Ausfuehrung am echten Edge-Standort nachweisen

Ein erfolgreicher ARM-Deploy bedeutet noch nicht, dass die neue Route am PoP aktiv ist.
Dieser GET muss durch die Action mit HTTP 403 und x-upload-policy: v1 abgelehnt werden.
Bei HTTP 404 oder fehlendem Header zuerst die Front-Door-Propagation abwarten
und diese Zelle erneut ausfuehren. Vorher keine Uploadzellen starten.

In [ ]:
http --ignore-stdin --timeout=30 --pretty=none --print=hb --cert="$CERT" --cert-key="$KEY" \
  GET "$EDGE_URL/readiness.json" | tee "$DEMO_DIR/readiness.response" &&
grep -q '^HTTP/.* 403 ' "$DEMO_DIR/readiness.response" &&
grep -qi '^x-upload-policy: v1' "$DEMO_DIR/readiness.response"


## Testdateien vorbereiten

Telemetry: JSON bis 1 MiB. Diagnostics: ZIP bis 20 MiB.
Die zu grosse Datei hat exakt 1 MiB + 1 Byte. Python wird nur zum Erstellen des ZIP-Archivs verwendet.

In [ ]:
mkdir -p "$DEMO_DIR" &&
jq -n --arg device "$DEVICE_ID" '{device:$device,temperature:23.5}' > "$TELEMETRY_FILE" &&
python3 -m zipfile -c "$DIAGNOSTICS_FILE" "$TELEMETRY_FILE" &&
truncate -s 1048577 "$OVERSIZE_FILE" &&
wc -c "$TELEMETRY_FILE" "$DIAGNOSTICS_FILE" "$OVERSIZE_FILE"


## Telemetry hochladen und gefaelschte Metadaten ueberschreiben

Erwartet: HTTP 201. Die Action ersetzt die drei mitgesendeten Metadatenwerte.
`--check-status` laesst unerwartete HTTP-Fehler die Zelle fehlschlagen.

In [ ]:
http --timeout=60 --check-status --pretty=none --print=hb --cert="$CERT" --cert-key="$KEY" \
  PUT "$EDGE_URL/$RUN_ID.json" \
  Content-Type:application/json X-Upload-Kind:telemetry X-Device-Id:"$DEVICE_ID" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  x-ms-meta-policy_version:forged x-ms-meta-device_id:other-device x-ms-meta-upload_kind:forged \
  < "$TELEMETRY_FILE" | tee "$DEMO_DIR/telemetry.response" &&
grep -q '^HTTP/.* 201 ' "$DEMO_DIR/telemetry.response"


In [ ]:
az storage blob show --subscription "$AZURE_SUBSCRIPTION_ID" --auth-mode login \
  --account-name "$STORAGE" --container-name "$CONTAINER" --name "$TELEMETRY_BLOB" |
jq -e --arg device "$DEVICE_ID" '.metadata | select(.policy_version == "v1" and .device_id == $device and .upload_kind == "telemetry")'


## Diagnostics hochladen

Erwartet: HTTP 201 und Uploadtyp diagnostics in den Blob-Metadaten.

In [ ]:
http --timeout=60 --check-status --pretty=none --print=hb --cert="$CERT" --cert-key="$KEY" \
  PUT "$EDGE_URL/$RUN_ID.zip" \
  Content-Type:application/zip X-Upload-Kind:diagnostics X-Device-Id:"$DEVICE_ID" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  < "$DIAGNOSTICS_FILE" | tee "$DEMO_DIR/diagnostics.response" &&
grep -q '^HTTP/.* 201 ' "$DEMO_DIR/diagnostics.response"


In [ ]:
az storage blob show --subscription "$AZURE_SUBSCRIPTION_ID" --auth-mode login \
  --account-name "$STORAGE" --container-name "$CONTAINER" --name "$DIAGNOSTICS_BLOB" |
jq -e '.metadata | select(.upload_kind == "diagnostics" and .policy_version == "v1")'


## Falscher Content-Type

Erwartet: HTTP 403 und file-type-mismatch. Negative HTTP-Tests verzichten bewusst
auf `--check-status`; stattdessen pruefen grep-Aufrufe den erwarteten Status und
den konkreten Fehlerheader. Netzwerkfehler bleiben durch pipefail Fehler.

In [ ]:
http --timeout=60 --pretty=none --print=hb --cert="$CERT" --cert-key="$KEY" \
  PUT "$EDGE_URL/$RUN_ID-type.json" \
  Content-Type:text/plain X-Upload-Kind:telemetry X-Device-Id:"$DEVICE_ID" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  < "$TELEMETRY_FILE" | tee "$DEMO_DIR/type.response" &&
grep -q '^HTTP/.* 403 ' "$DEMO_DIR/type.response" &&
grep -qi '^x-upload-error: file-type-mismatch' "$DEMO_DIR/type.response"


## Falsche Dateiendung

Erwartet: HTTP 403 und file-type-mismatch, weil Telemetry als ZIP benannt wurde.

In [ ]:
http --timeout=60 --pretty=none --print=hb --cert="$CERT" --cert-key="$KEY" \
  PUT "$EDGE_URL/$RUN_ID-extension.zip" \
  Content-Type:application/json X-Upload-Kind:telemetry X-Device-Id:"$DEVICE_ID" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  < "$TELEMETRY_FILE" | tee "$DEMO_DIR/extension.response" &&
grep -q '^HTTP/.* 403 ' "$DEMO_DIR/extension.response" &&
grep -qi '^x-upload-error: file-type-mismatch' "$DEMO_DIR/extension.response"


## Unbekannter Uploadtyp

Erwartet: HTTP 403 und unknown-upload-kind.

In [ ]:
http --timeout=60 --pretty=none --print=hb --cert="$CERT" --cert-key="$KEY" \
  PUT "$EDGE_URL/$RUN_ID-kind.json" \
  Content-Type:application/json X-Upload-Kind:firmware X-Device-Id:"$DEVICE_ID" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  < "$TELEMETRY_FILE" | tee "$DEMO_DIR/kind.response" &&
grep -q '^HTTP/.* 403 ' "$DEMO_DIR/kind.response" &&
grep -qi '^x-upload-error: unknown-upload-kind' "$DEMO_DIR/kind.response"


## Fehlende Geraete-ID

Erwartet: HTTP 403 und invalid-device-id.

In [ ]:
http --timeout=60 --pretty=none --print=hb --cert="$CERT" --cert-key="$KEY" \
  PUT "$EDGE_URL/$RUN_ID-device.json" \
  Content-Type:application/json X-Upload-Kind:telemetry \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  < "$TELEMETRY_FILE" | tee "$DEMO_DIR/device.response" &&
grep -q '^HTTP/.* 403 ' "$DEMO_DIR/device.response" &&
grep -qi '^x-upload-error: invalid-device-id' "$DEMO_DIR/device.response"


## Deklarierte Groesse ueberschritten

Erwartet: HTTP 403 und declared-size-exceeded. HTTPie setzt Content-Length passend
zur Datei mit 1048577 Bytes. Wir senden keinen absichtlich falschen Laengenheader.

In [ ]:
http --timeout=60 --pretty=none --print=hb --cert="$CERT" --cert-key="$KEY" \
  PUT "$EDGE_URL/$RUN_ID-size.json" \
  Content-Type:application/json X-Upload-Kind:telemetry X-Device-Id:"$DEVICE_ID" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  < "$OVERSIZE_FILE" | tee "$DEMO_DIR/size.response" &&
grep -q '^HTTP/.* 403 ' "$DEMO_DIR/size.response" &&
grep -qi '^x-upload-error: declared-size-exceeded' "$DEMO_DIR/size.response"


## Falsche HTTP-Methode

Erwartet: HTTP 403, method-not-allowed und Allow: PUT. Lesen erfolgt in diesem Showcase direkt
mit Entra-Berechtigungen am Storage, nicht ueber die Uploadroute.

In [ ]:
http --ignore-stdin --timeout=60 --pretty=none --print=hb --cert="$CERT" --cert-key="$KEY" \
  GET "$EDGE_URL/$RUN_ID.json" | tee "$DEMO_DIR/method.response" &&
grep -q '^HTTP/.* 403 ' "$DEMO_DIR/method.response" &&
grep -qi '^allow: PUT' "$DEMO_DIR/method.response"


## Ohne Zertifikat bleibt mTLS wirksam

Front Door lehnt schon vor der Action mit einem HTTP-Clientfehler ab.
Die Folgezelle prueft zusaetzlich, dass kein Blob entstanden ist.
DNS- und Verbindungsfehler zaehlen nicht als erfolgreicher mTLS-Test.

In [ ]:
http --timeout=30 --pretty=none --print=hb \
  PUT "$EDGE_URL/$RUN_ID-no-cert.json" \
  Content-Type:application/json X-Upload-Kind:telemetry X-Device-Id:"$DEVICE_ID" \
  x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  < "$TELEMETRY_FILE" | tee "$DEMO_DIR/mtls.response" &&
grep -q '^HTTP/.* 4[0-9][0-9] ' "$DEMO_DIR/mtls.response"


In [ ]:
az storage blob exists --subscription "$AZURE_SUBSCRIPTION_ID" --auth-mode login \
  --account-name "$STORAGE" --container-name "$CONTAINER" --name "edge/$RUN_ID-no-cert.json" |
jq -e '.exists == false'


## Vergleich: bestehender Pfad ohne Edge Action

Ohne Uploadtyp oder Geraete-ID funktioniert `/upload/*` weiterhin.
Erwartet: HTTP 201 und keine policy_version in den Metadaten.

In [ ]:
http --timeout=60 --check-status --pretty=none --print=hb --cert="$CERT" --cert-key="$KEY" \
  PUT "https://$HOST/$CONTAINER/$BASELINE_BLOB" \
  Content-Type:text/plain x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 \
  < "$TELEMETRY_FILE" | tee "$DEMO_DIR/baseline.response" &&
grep -q '^HTTP/.* 201 ' "$DEMO_DIR/baseline.response"


In [ ]:
az storage blob show --subscription "$AZURE_SUBSCRIPTION_ID" --auth-mode login \
  --account-name "$STORAGE" --container-name "$CONTAINER" --name "$BASELINE_BLOB" |
jq -e '.metadata | has("policy_version") | not'


## Nachweisen, dass Ablehnungen keine Blobs erzeugt haben

Unter dem laufbezogenen Edge-Praefix duerfen nur das JSON und das ZIP existieren.

In [ ]:
az storage blob list --subscription "$AZURE_SUBSCRIPTION_ID" --auth-mode login \
  --account-name "$STORAGE" --container-name "$CONTAINER" --prefix "edge/$RUN_ID" |
jq -e --arg json "$TELEMETRY_BLOB" --arg zip "$DIAGNOSTICS_BLOB" \
  'map(.name) | sort | select(. == ([$json, $zip] | sort))'


## Mit AzCopy auflisten und herunterladen

AzCopy verwendet `AZCOPY_AUTO_LOGIN_TYPE=AZCLI` direkt am Storage. Es verwendet
weder ein SAS noch das Clientzertifikat und fuehrt die Edge Action nicht aus.
Fuer die Ruecklesepruefung braucht dein angemeldetes Konto Blob-Datenrechte.

In [ ]:
azcopy list "https://$STORAGE.blob.core.windows.net/$CONTAINER/edge" --machine-readable

In [ ]:
azcopy copy "https://$STORAGE.blob.core.windows.net/$CONTAINER/$TELEMETRY_BLOB" "$DEMO_DIR/downloaded.json" &&
cmp "$TELEMETRY_FILE" "$DEMO_DIR/downloaded.json" &&
echo "Telemetry unveraendert gespeichert"


In [ ]:
azcopy copy "https://$STORAGE.blob.core.windows.net/$CONTAINER/$DIAGNOSTICS_BLOB" "$DEMO_DIR/downloaded.zip" &&
cmp "$DIAGNOSTICS_FILE" "$DEMO_DIR/downloaded.zip" &&
python3 -m zipfile -l "$DEMO_DIR/downloaded.zip"


## Testdaten dieses Laufs entfernen

Die drei explizit benannten Blobs werden geloescht. Andere Uploads bleiben erhalten.
Action, Route und bestehende Infrastruktur werden nicht entfernt.

In [ ]:
az storage blob delete --subscription "$AZURE_SUBSCRIPTION_ID" --auth-mode login \
  --account-name "$STORAGE" --container-name "$CONTAINER" --name "$TELEMETRY_BLOB" &&
az storage blob delete --subscription "$AZURE_SUBSCRIPTION_ID" --auth-mode login \
  --account-name "$STORAGE" --container-name "$CONTAINER" --name "$DIAGNOSTICS_BLOB" &&
az storage blob delete --subscription "$AZURE_SUBSCRIPTION_ID" --auth-mode login \
  --account-name "$STORAGE" --container-name "$CONTAINER" --name "$BASELINE_BLOB"


In [ ]:
az storage blob list --subscription "$AZURE_SUBSCRIPTION_ID" --auth-mode login \
  --account-name "$STORAGE" --container-name "$CONTAINER" --prefix "edge/$RUN_ID" |
jq -e 'length == 0'


In [ ]:
rm "$TELEMETRY_FILE" "$DIAGNOSTICS_FILE" "$OVERSIZE_FILE" \
  "$DEMO_DIR/downloaded.json" "$DEMO_DIR/downloaded.zip" \
  "$DEMO_DIR/telemetry.response" "$DEMO_DIR/diagnostics.response" \
  "$DEMO_DIR/type.response" "$DEMO_DIR/extension.response" "$DEMO_DIR/kind.response" \
  "$DEMO_DIR/device.response" "$DEMO_DIR/size.response" "$DEMO_DIR/method.response" \
  "$DEMO_DIR/baseline.response" "$DEMO_DIR/mtls.response" "$DEMO_DIR/readiness.response" &&
echo "Demo-Dateien entfernt. AzCopy-Logs verbleiben zur Diagnose in $DEMO_DIR."
